# FineWeb2 vs. Lang2Vec coverage

Loads the FineWeb2 distribution metadata and Lang2Vec inventory to quantify how many languages overlap between the two resources.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.manifold import MDS
from sklearn.metrics import silhouette_samples, silhouette_score
from itertools import combinations


def find_repo_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / "data_prep").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")


try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

ROOT = find_repo_root(NOTEBOOK_DIR)
FW_LANG_PATH = ROOT / "data_prep" / "base_data" / "fineweb2-language-distribution.csv"
LANG2VEC_DIR = ROOT / "data_prep" / "lang_cluster_analysis" / "lang2vec"
sys.path.insert(0, str(LANG2VEC_DIR))
from lang2vec import lang2vec as l2v

fw_df = pd.read_csv(FW_LANG_PATH, usecols=["code", "family"])  # duplicates per split/subset
fw_unique = fw_df.drop_duplicates("code").reset_index(drop=True)
fw_langs = fw_unique["code"].tolist()
try:
    l2v_langs = sorted(l2v.DISTANCE_LANGUAGES)
except AttributeError:
    if hasattr(l2v, "available_distance_languages"):
        l2v_langs = sorted(l2v.available_distance_languages())
    else:
        raise
overlap = sorted(set(fw_langs) & set(l2v_langs))
missing = sorted(set(fw_langs) - set(l2v_langs))
extra = sorted(set(l2v_langs) - set(fw_langs))
family_map = dict(zip(fw_unique["code"], fw_unique["family"]))
overlap_langs = overlap
overlap_families = [family_map[lang] for lang in overlap_langs]
family_coverage = (
    fw_unique.assign(in_lang2vec=fw_unique["code"].isin(overlap))
    .groupby("family")
    .agg(total_langs=("code", "count"), covered=("in_lang2vec", "sum"))
    .assign(coverage_pct=lambda df: (df["covered"] / df["total_langs"] * 100).round(1))
    .sort_values("total_langs", ascending=False)
)
top_families = family_coverage.head(5)
coverage_msg = ", ".join(
    [
        f"{idx}: {row.covered}/{row.total_langs} ({row.coverage_pct}%)"
        for idx, row in top_families.iterrows()
    ]
)
summary = {
    "fineweb_total": len(fw_langs),
    "lang2vec_total": len(l2v_langs),
    "overlap": len(overlap),
    "fineweb_coverage_pct": round(len(overlap) / len(fw_langs) * 100, 1),
    "lang2vec_coverage_pct": round(len(overlap) / len(l2v_langs) * 100, 1),
}
summary


In [ ]:
print(
    f"Overlap: {summary['overlap']} langs (covers {summary['fineweb_coverage_pct']}% of FineWeb2 and {summary['lang2vec_coverage_pct']}% of Lang2Vec)."
)
print(f"Most common families covered: {coverage_msg}.")
if missing:
    print(
        f"FineWeb2 langs missing from Lang2Vec (first 10 of {len(missing)}): {', '.join(missing[:10])}..."
    )
else:
    print("Lang2Vec covers every FineWeb2 language.")
if extra:
    print(
        f"Lang2Vec-only languages (first 10 of {len(extra)}): {', '.join(extra[:10])}..."
    )
else:
    print("Every Lang2Vec language appears in FineWeb2.")

In [ ]:
distance_type = "genetic"
distance_matrix = np.asarray(l2v.distance(distance_type, overlap_langs), dtype=float)
np.fill_diagonal(distance_matrix, 0.0)
{
    "distance_type": distance_type,
    "matrix_shape": distance_matrix.shape,
}


In [ ]:
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=0, normalized_stress="auto")
coords = mds.fit_transform(distance_matrix)
family_palette = {fam: idx for idx, fam in enumerate(sorted(set(overlap_families)))}
colors = [family_palette[fam] for fam in overlap_families]
cmap = plt.cm.get_cmap("tab20", max(len(family_palette), 1))
plt.figure(figsize=(11, 8))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=colors, cmap=cmap, s=45, alpha=0.85, edgecolor="k", linewidth=0.25)
for (x, y), lang, fam in zip(coords, overlap_langs, overlap_families):
    plt.text(x, y, f"{lang} ({fam})", fontsize=6, ha="center", va="center", alpha=0.7)
legend_families = list(family_palette.keys())[:15]
handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color=scatter.cmap(scatter.norm(family_palette[fam])), label=fam)
    for fam in legend_families
]
if len(family_palette) > len(legend_families):
    handles.append(plt.Line2D([0], [0], marker="o", linestyle="", color="gray", label="(others)"))
if handles:
    plt.legend(handles=handles, title="Family", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.title(f"Lang2Vec {distance_type} distances (MDS, colored by family)")
plt.xlabel("MDS dim 1")
plt.ylabel("MDS dim 2")
plt.tight_layout()
plt.show()


In [ ]:
family_counts = pd.Series(overlap_families).value_counts()
valid_indices = np.array([idx for idx, fam in enumerate(overlap_families) if family_counts[fam] >= 2])
multi_families = sorted(family_counts[family_counts >= 2].index.tolist())
if len(valid_indices) < 2 or len(multi_families) < 2:
    print("Not enough multi-language families to compute silhouettes.")
    family_sil_df = pd.DataFrame({"code": overlap_langs, "family": overlap_families, "silhouette": np.nan})
    overall_family_silhouette = np.nan
else:
    sub_matrix = distance_matrix[np.ix_(valid_indices, valid_indices)]
    label_map = {fam: idx for idx, fam in enumerate(multi_families)}
    sub_labels = [label_map[overlap_families[idx]] for idx in valid_indices]
    sil_values = silhouette_samples(sub_matrix, sub_labels, metric="precomputed")
    sil_vector = np.full(len(overlap_langs), np.nan)
    sil_vector[valid_indices] = sil_values
    family_sil_df = pd.DataFrame({"code": overlap_langs, "family": overlap_families, "silhouette": sil_vector})
    overall_family_silhouette = silhouette_score(sub_matrix, sub_labels, metric="precomputed")
    print(f"Overall silhouette treating families as clusters: {overall_family_silhouette:.3f}")
family_sil_df.head()


In [ ]:
if family_sil_df['silhouette'].notna().sum() == 0:
    print("No valid silhouette scores to summarize.")
else:
    family_sil_summary = (
        family_sil_df.dropna()
        .groupby('family')
        .agg(count=('code', 'count'), mean_sil=('silhouette', 'mean'), median_sil=('silhouette', 'median'))
        .assign(mean_sil=lambda df: df['mean_sil'].round(3), median_sil=lambda df: df['median_sil'].round(3))
        .sort_values('mean_sil', ascending=False)
    )
    display(family_sil_summary)


In [ ]:
family_to_indices = {}
for idx, fam in enumerate(overlap_families):
    family_to_indices.setdefault(fam, []).append(idx)
families = sorted(family_to_indices)
pairwise_family_distance = {}
for i, fam_a in enumerate(families):
    idx_a = family_to_indices[fam_a]
    for j in range(i + 1, len(families)):
        fam_b = families[j]
        idx_b = family_to_indices[fam_b]
        avg_dist = float(distance_matrix[np.ix_(idx_a, idx_b)].mean())
        pairwise_family_distance[(fam_a, fam_b)] = avg_dist
best_combo = None
best_score = -1.0
for combo in combinations(families, 4):
    pair_dists = [
        pairwise_family_distance[tuple(sorted((fam_a, fam_b)))]
        for fam_a, fam_b in combinations(combo, 2)
    ]
    avg_pair = float(np.mean(pair_dists))
    if avg_pair > best_score:
        best_score = avg_pair
        best_combo = combo
most_distinct_families = list(best_combo)
distinct_pair_details = {
    f"{fam_a} vs {fam_b}": round(
        pairwise_family_distance[tuple(sorted((fam_a, fam_b)))], 3
    )
    for fam_a, fam_b in combinations(most_distinct_families, 2)
}
{
    "families": most_distinct_families,
    "avg_pairwise_distance": round(best_score, 3),
    "pair_distances": distinct_pair_details,
}


In [ ]:
top_family_set = set(most_distinct_families)
top_indices = [idx for idx, fam in enumerate(overlap_families) if fam in top_family_set]
subset_langs = [overlap_langs[idx] for idx in top_indices]
subset_families = [overlap_families[idx] for idx in top_indices]
subset_matrix = distance_matrix[np.ix_(top_indices, top_indices)]
subset_mds = MDS(n_components=2, dissimilarity="precomputed", random_state=0, normalized_stress="auto")
subset_coords = subset_mds.fit_transform(subset_matrix)
subset_palette = {fam: idx for idx, fam in enumerate(sorted(top_family_set))}
subset_colors = [subset_palette[fam] for fam in subset_families]
subset_cmap = plt.cm.get_cmap("tab10", len(subset_palette))
plt.figure(figsize=(9, 6))
subset_scatter = plt.scatter(subset_coords[:, 0], subset_coords[:, 1], c=subset_colors, cmap=subset_cmap, s=60, alpha=0.9, edgecolor="k", linewidth=0.3)
for (x, y), lang, fam in zip(subset_coords, subset_langs, subset_families):
    plt.text(x, y, f"{lang} ({fam})", fontsize=7, ha="center", va="center", alpha=0.75)
handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color=subset_scatter.cmap(subset_scatter.norm(subset_palette[fam])), label=fam)
    for fam in sorted(top_family_set)
]
plt.legend(handles=handles, title="Family", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("Most distinct language families (Lang2Vec distances)")
plt.xlabel("MDS dim 1")
plt.ylabel("MDS dim 2")
plt.tight_layout()
plt.show()


In [ ]:
from collections import defaultdict

distance_type = 'genetic'
D_lang = np.asarray(l2v.distance(distance_type, overlap_langs), dtype=float)
np.fill_diagonal(D_lang, np.inf)
family_idx = defaultdict(list)
for idx, lang in enumerate(overlap_langs):
    family_idx[family_map[lang]].append(idx)
families = list(family_idx)
def family_distance(fa, fb):
    I, J = family_idx[fa], family_idx[fb]
    return float(D_lang[np.ix_(I, J)].mean())
F = np.array([[family_distance(f1, f2) for f2 in families] for f1 in families])
selected = [0]
while len(selected) < min(5, len(families)):
    remaining = [i for i in range(len(families)) if i not in selected]
    if not remaining:
        break
    pick = max(remaining, key=lambda i: min(F[i, j] for j in selected))
    selected.append(pick)
distinct_families_simple = [families[i] for i in selected]
def closest_pair(idxs):
    sub = D_lang[np.ix_(idxs, idxs)].copy()
    np.fill_diagonal(sub, np.inf)
    flat_idx = np.argmin(sub)
    i, j = divmod(flat_idx, sub.shape[1])
    return overlap_langs[idxs[i]], overlap_langs[idxs[j]], sub[i, j]
similar_langs = {fam: closest_pair(family_idx[fam]) for fam in distinct_families_simple}
print('Greedy distinct families:', distinct_families_simple)
for fam, (a, b, dist) in similar_langs.items():
    print(f"{fam}: closest pair {a}-{b} distance {dist:.3f}")
